# 03. Концептуальная схема переразмещения целых подграфов

Самостоятельная концептуальная часть TASK R1.4, §7 и раздела 6 будущей статьи. **Это не действующий контроллер, не найденный допустимый план и не эксперимент сравнения методов.** Эталон `v2/` читается, не изменяется и не запускается.

In [ ]:
from pathlib import Path
import os, sys, json, subprocess
root = Path.cwd()
while root != root.parent and not (root/'efc/companion.py').exists():
    root = root.parent
if not (root/'efc/companion.py').exists():
    try:
        import google.colab
    except ImportError:
        raise RuntimeError('Запустите из копии night/fefc-v3 или в Colab.')
    root = Path('/content/fefc-r14-companions')
    if not root.exists():
        subprocess.run(['git','clone','--single-branch','--branch','night/fefc-v3',
            'https://github.com/annfridreksen/Forecast_Edge_Fog_Cloud.git',str(root)],check=True)
    if not (root/'efc/companion.py').exists():
        raise RuntimeError('Копия устарела: companion.py отсутствует. Молчаливого обновления нет.')
    subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(root/'requirements.txt')],check=True)
os.chdir(root)
sys.path.insert(0,str(root))
SOURCE_SHA = subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip() if (root/'.git').exists() else os.environ.get('FEFC_SOURCE_SHA','архив: SHA в реестре')
from IPython.display import display
from IPython import get_ipython
get_ipython().run_line_magic('matplotlib','inline')
import matplotlib.pyplot as plt
import pandas as pd
from efc import companion as c
print('Источник:',SOURCE_SHA,'; Python:',sys.version.split()[0])
print('Только чтение и схемы: имитация, обучение, прогноз и применение плана не запускаются.')

## 1. Неизменённый критерий эталона

Для допустимых планов приоритет: **(R,T,N,id)**. R — вектор порядковых команд, отсортированный по убыванию; сравнение лексикографическое, меньшее предпочтительнее. Затем — меньшее расчётное время T, число переназначений N, идентификатор. При неприменимых командах используется базовый режим по текущим данным. Ниже проверяются байты защищённого исходника по его manifest и показываются функции, без их исполнения.

In [ ]:
reference = c.reference_fragments(root)
print(reference['path'],reference['sha256'])
for name,text in reference['fragments'].items():
    print('\n'+text)
assert reference['executed'] is False

## 2. «До/после» — только условная структура

A={a1,a2}, B={b}, C={c}; зависимости A→B→C. Буквы E/F обозначают условные краевые/туманные устройства. Переназначается целый B; a1 и a2 нельзя назначать независимо внутри A. Состав частей не меняется, переразбиение не выполняется. Cloud присутствует в среде, но не входит в кандидаты этого метода.

Численные профили, ресурсные ограничения, маршруты и срок для этой схемы не заданы. **Допустимость не проверена.** Проверка следующей ячейки относится только к составу и полноте символических назначений.

In [ ]:
spec = c.symbolic_placement()
assert c.check_symbolic(spec)
display(pd.DataFrame([{'Подграф':b,'Вершины':', '.join(v),'До':spec['before'][b],'Условно после':spec['after'][b]} for b,v in spec['members'].items()]))
print('Проверена структура рисунка; НЕ ресурсы, НЕ расписание и НЕ допустимость плана.')

## 3. Предпочтение отделено от проверки полного плана

Каждое полное назначение требует проверки ресурсов, информационных зависимостей, передач и срока всей связной задачи. Прогноз не проверяет ограничения и не выбирает план самостоятельно. Ни изменение числа Fog, ни увеличение буфера не равны переразмещению подграфов.

In [ ]:
fig = c.concept_figure('placement')
display(fig); plt.close(fig)

## 4. Границы

Ограниченный поиск без результата не доказывает невозможность размещения. Расчётное время не является гарантированным сроком. Для применения плана потребуются актуальный снимок, сохранность входов и отдельные правила перехода; живая миграция здесь не реализована. Прогнозная разгрузка Fog→Cloud относится к дальнейшей работе.

| Утверждение | Проверка |
|---|---|
| Критерий эталона не изменён | `r14-03-reference`: фактический SHA файла и manifest |
| Части целые и состав сохранён | `r14-03-symbolic`: проверка только структуры схемы |
| Интеграция прогноза и управления | Не выполнена, экспериментальных чисел нет |

In [ ]:
print(json.dumps(dict(notebook='03',source=SOURCE_SHA,reference_sha256=reference['sha256'],reference_executed=False,symbolic_structure_checked=True,full_plan_verified=False,simulation_executed=False,forecast_executed=False,training_executed=False,controller_executed=False,colab_browser_checked=False),ensure_ascii=False,indent=2))